In [5]:
import os
import glob
import MDAnalysis as mda

base_dir = "/deac/phy/classes/phy320-fa-2026/andrtj24/Project-1-PHY320620-Fall2026/6od3-1"
if not os.path.exists(base_dir):
    base_dir = "/deac/phy/classes/phy320-fa-2026/andrtj24/Project-1-PHY320620-Fall2026/6o3-1"

# Modified to only include '1'
replica_dirs = ['1'] 
universes = {}

for rep in replica_dirs:
    target_dir = os.path.join(base_dir, rep, "namd")
    psf_path = glob.glob(os.path.join(target_dir, "*.psf"))
    dcd_path = sorted(glob.glob(os.path.join(target_dir, "*.dcd")))
    
    if psf_path and dcd_path:
        universes[rep] = mda.Universe(psf_path[0], dcd_path)
        print(f"Loaded Replica {rep}: {len(universes[rep].trajectory)} frames.")

if not universes:
    print("Error: Could not load the trajectory.")

/home/andrtj24/.local/lib/python3.11/site-packages/MDAnalysis/coordinates/DCD.py:171: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


Loaded Replica 1: 40127 frames.


In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import rms
# Import HydrogenBondAnalysis from its correct MDAnalysis 2.x location
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis
import matplotlib.pyplot as plt
import numpy as np

# Assuming your MDAnalysis Universe is already loaded as 'u'.
print(f"Running advanced analysis on {len(u.trajectory)} frames...")

# 1. Conformational Stability: RMSD of Protein and DNA
print("Calculating RMSD for conformational stability...")
rmsd_analysis = rms.RMSD(
    u, 
    u, 
    select="protein and backbone",           # Align by protein backbone
    groupselections=["nucleic and backbone"] # Calculate RMSD for DNA backbone
)
rmsd_analysis.run()

# 2. Structural Flexibility: RMSF of the TCF4 C-alpha atoms
print("Calculating RMSF for protein flexibility...")
calphas = u.select_atoms("protein and name CA")
rmsf_analysis = rms.RMSF(calphas).run()

# 3. Interface Architecture: Protein-DNA Hydrogen Bonds
print("Analyzing Protein-DNA Hydrogen Bond network...")
# Updated: 'between' now requires a nested list of selection strings
hbond_analysis = HydrogenBondAnalysis(
    universe=u,
    between=[["protein", "nucleic"]], 
    d_a_cutoff=3.0,          # Distance cutoff in Angstroms
    d_h_a_angle_cutoff=150   # Angle cutoff in degrees
)
hbond_analysis.run()

# ==========================================
# Visualization of Advanced Analysis Results
# ==========================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: RMSD (Stability)
# Assuming a standard timestep; adjust the division factor if your saving interval differs
time_ns = rmsd_analysis.results.rmsd[:, 1] / 1000  
axes[0].plot(time_ns, rmsd_analysis.results.rmsd[:, 2], label="TCF4 Backbone", color='blue')
axes[0].plot(time_ns, rmsd_analysis.results.rmsd[:, 3], label="DNA Backbone", color='orange')
axes[0].set_title("Conformational Stability (RMSD)")
axes[0].set_xlabel("Frame / Time")
axes[0].set_ylabel("RMSD (Å)")
axes[0].legend()

# Plot 2: RMSF (Flexibility)
axes[1].plot(calphas.resids, rmsf_analysis.results.rmsf, color='green')
axes[1].set_title("TCF4 Residue Flexibility (RMSF)")
axes[1].set_xlabel("Residue Number")
axes[1].set_ylabel("RMSF (Å)")

# Plot 3: Hydrogen Bonds (Interface)
axes[2].plot(hbond_analysis.times, hbond_analysis.count_by_time(), color='purple', alpha=0.5)
axes[2].set_title("Protein-DNA Interface H-Bonds")
axes[2].set_xlabel("Frame / Time")
axes[2].set_ylabel("Number of Hydrogen Bonds")

plt.tight_layout()
plt.savefig("advanced_structural_analysis.png", dpi=300, bbox_inches='tight')
plt.show()

# Output summary metric
avg_hbonds = np.mean(hbond_analysis.count_by_time())
print(f"Average number of direct Protein-DNA hydrogen bonds: {avg_hbonds:.2f}")


Running advanced analysis on 40127 frames...
Calculating RMSD for conformational stability...
Calculating RMSF for protein flexibility...
Analyzing Protein-DNA Hydrogen Bond network...
